# 04. Подготовка единой таблицы профилей для Entity Resolution

Цель этого шага — **не candidate generation** и **не pair dataset**.

Здесь мы приводим исходный датасет к прозрачному аналитическому виду:

1. Берём исходные строки событий/профилей.
2. Разворачиваем массивы `non_processing_features` и `fs_features`.
3. Разворачиваем JSON-поле `realtime_features`.
4. Собираем единую широкую таблицу уровня `profile_id`.
5. Сохраняем длинные token-level таблицы для аудита.
6. Готовим отчёт по колонкам: какие признаки подходят для blocking, какие лучше оставить для анализа качества или последующих моделей.

После этого уже можно осознанно переходить к **candidate generation + pair dataset**.

In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
DATA_PATH = Path("../data/raw")

INPUT_PATH =  DATA_PATH / "split_label_train_V3.snappy.parquet"
OUTPUT_DIR = DATA_PATH / "prepared_entity_resolution_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUTPUT_DIR

## 1. Загружаем исходный датасет

Важно: используем именно исходный датасет `split_label_train_V3.snappy.parquet`, потому что в агрегированном `profile_level_eda.parquet` поля `non_processing_features`, `fs_features`, `realtime_features` уже потеряны как сырой источник.

In [ ]:
raw = pd.read_parquet(INPUT_PATH).reset_index(names='raw_row_id')
raw['created_at'] = pd.to_datetime(raw['created_at'], errors='coerce')

print(raw.shape)
display(raw.head(3))
display(raw.dtypes)

In [ ]:
# Базовая проверка объёма данных
summary_input = {
    'raw_rows': len(raw),
    'profiles': raw['profile_id'].nunique(),
    'entities': raw['entity_id'].nunique(),
    'columns': raw.shape[1],
}
summary_input

## 2. Вспомогательные функции

Нам нужны три типа преобразований:

- нормализация обычных идентификационных полей: имя, email, телефон, дата рождения;
- разворот массивов вида `feature:value`;
- разворот JSON-поля `realtime_features`.

In [ ]:
EMPTY = {'', 'none', 'null', 'nan', '<na>'}


def is_empty(x) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    return isinstance(x, str) and x.strip().lower() in EMPTY


def mode_non_empty(s: pd.Series):
    vals = [str(x).strip() for x in s if not is_empty(x)]
    if not vals:
        return pd.NA
    c = Counter(vals)
    return sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

def normalize_text(s: pd.Series) -> pd.Series:
    return (
        s.astype('string')
         .str.strip()
         .str.lower()
         .replace({'': pd.NA, 'none': pd.NA, 'nan': pd.NA, 'null': pd.NA})
    )


def digits_only(x):
    if is_empty(x):
        return pd.NA
    d = re.sub(r'\D+', '', str(x))
    return d if d else pd.NA

## 3. Базовая таблица уровня `profile_id`

Одна строка = один `profile_id`.

Здесь пока нет blocking. Это минимальная аналитическая основа:

- сколько событий у профиля и когда они были;
- к какому `entity_id` он относится как к ground truth;
- размер настоящей группы по `entity_id` для оценки качества;
- один нормализованный вариант identity-полей, который можно использовать в анализе и blocking.

In [ ]:
def build_profile_base(raw: pd.DataFrame) -> pd.DataFrame:
    g = raw.groupby('profile_id', dropna=False, sort=False)

    base = g.agg(
        event_count=('raw_row_id', 'size'),
        entity_id=('entity_id', mode_non_empty),
        first_event_at=('created_at', 'min'),
        last_event_at=('created_at', 'max'),
        first_name=('first_name', mode_non_empty),
        last_name=('last_name', mode_non_empty),
        email=('email', mode_non_empty),
        phone=('phone', mode_non_empty),
        birthday=('birthday', mode_non_empty),
        sex=('sex', mode_non_empty),
    ).reset_index()

    entity_size = (
        base.groupby('entity_id', dropna=False)['profile_id']
            .nunique()
            .rename('entity_size')
            .reset_index()
    )
    base = base.merge(entity_size, on='entity_id', how='left')
    base['first_name_norm'] = normalize_text(base['first_name'])
    base['last_name_norm'] = normalize_text(base['last_name'])
    base['sex_norm'] = normalize_text(base['sex']).replace({'unknown': pd.NA})
    base['email_norm'] = normalize_text(base['email'])
    base['email_domain'] = base['email_norm'].str.extract(r'@(.+)$', expand=False)

    base['phone_digits'] = base['phone'].map(digits_only).astype('string')

    birthday_dt = pd.to_datetime(base['birthday'], errors='coerce')
    base['birthday_ymd'] = birthday_dt.dt.strftime('%Y-%m-%d').astype('string')

    identity_cols = [
        'first_name_norm', 'last_name_norm', 'sex_norm',
        'email_norm', 'phone_digits', 'birthday_ymd'
    ]
    # простая техническая метрика: сколько identity-полей заполнено у профиля.
    base['identity_filled_count'] = base[identity_cols].notna().sum(axis=1).astype('int16')

    keep_cols = [
        'profile_id', 'event_count', 'entity_id', 'entity_size',
        'first_event_at', 'last_event_at',
        'first_name_norm', 'last_name_norm', 'sex_norm',
        'email_norm', 'email_domain', 'phone_digits', 'birthday_ymd',
        'identity_filled_count',
    ]

    return base[keep_cols]


profile_base = build_profile_base(raw)
print(profile_base.shape)
display(profile_base.head(3))

## 4. Разворачиваем `non_processing_features` и `fs_features` в длинный формат

Длинный формат нужен для аудита:

- видно, какие токены были в исходных данных;
- можно посчитать покрытие признаков;
- можно проверить, какие признаки слишком редкие, слишком массовые или слишком высококардинальные;
- можно строить blocking-кандидаты не вслепую.

In [ ]:
def explode_array(raw: pd.DataFrame, col: str, source: str) -> pd.DataFrame:
    x = raw[['raw_row_id', 'profile_id', 'entity_id', 'created_at', col]].explode(col, ignore_index=True)
    x = x.dropna(subset=[col]).rename(columns={col: 'token'})
    x['token'] = x['token'].astype('string').str.strip()
    x = x[x['token'].notna() & x['token'].ne('')]

    parts = x['token'].str.split(':', n=1, expand=True)
    x['source'] = source
    x['feature_name'] = parts[0].astype('string').str.strip()
    x['feature_value'] = parts[1].astype('string').str.strip() if parts.shape[1] > 1 else pd.NA
    x.loc[x['feature_value'].eq(''), 'feature_value'] = pd.NA

    return x[[
        'raw_row_id', 'profile_id', 'entity_id', 'created_at',
        'source', 'feature_name', 'feature_value', 'token'
    ]]


np_long = explode_array(raw, 'non_processing_features', 'non_processing_features')
fs_long = explode_array(raw, 'fs_features', 'fs_features')

print('non_processing long:', np_long.shape)
print('fs long:', fs_long.shape)
display(np_long.head(3))
display(fs_long.head(3))

## 5. Разворачиваем `realtime_features`

`realtime_features` хранится как JSON-строка. Его тоже переводим в длинную таблицу `feature_name / feature_value`.

In [ ]:
def parse_realtime_payload(payload) -> dict:
    if is_empty(payload):
        return {}
    try:
        d = json.loads(payload) if isinstance(payload, str) else payload
    except Exception:
        return {}
    return d if isinstance(d, dict) else {}


def explode_realtime(raw: pd.DataFrame) -> pd.DataFrame:
    records = []
    cols = ['raw_row_id', 'profile_id', 'entity_id', 'created_at', 'realtime_features']
    for raw_row_id, profile_id, entity_id, created_at, payload in raw[cols].itertuples(index=False, name=None):
        d = parse_realtime_payload(payload)
        for k, v in d.items():
            if v is not None:
                records.append((
                    raw_row_id, profile_id, entity_id, created_at,
                    'realtime_features', str(k), str(v)
                ))
    return pd.DataFrame(
        records,
        columns=['raw_row_id', 'profile_id', 'entity_id', 'created_at', 'source', 'feature_name', 'feature_value']
    )


rt_long = explode_realtime(raw)
print('realtime long:', rt_long.shape)
display(rt_long.head())

## 6. Собираем минимальные широкие блоки признаков

Широкая таблица нужна только как удобный плоский слой для последующего анализа blocking.

Поэтому здесь оставляем только признаки, которые дальше реально участвуют в правилах или в sanity-check:

- гео/context: `geoname_id`, `subdivision_1_iso_code`, `geoid`, `geoname`;
- device/context: `device`, `osfamily`;
- поведенческие `fs`-ключи, которые использовались в blocking-экспериментах.

Счётчики `__cnt`, `__nunique`, флаги и остальные сырые признаки не добавляем в wide-слой: для аудита остаются token-level таблицы.

In [23]:
WIDE_FEATURES = {
    'np': ['device', 'geoname_id', 'osfamily', 'subdivision_1_iso_code'],
    'rt': ['geoid', 'geoname'],
    'fs': [
        'source_site_365', 'source_site_30',
        'visited_30', 'visited_365',
        'has_account', 'has_click_365', 'has_click_30',
        'has_accept_365', 'has_accept_30',
        'has_order_365', 'has_order_30', 'has_view_90',
    ],
}


def wide_selected_values(long_df: pd.DataFrame, prefix: str, features: list[str]) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame(columns=['profile_id'])

    val = long_df[
        long_df['feature_value'].notna()
        & long_df['feature_name'].isin(features)
    ][['profile_id', 'feature_name', 'feature_value']].copy()

    if val.empty:
        return pd.DataFrame(columns=['profile_id'])

    val['feature_value'] = val['feature_value'].astype(str)
    counts = (
        val.groupby(['profile_id', 'feature_name', 'feature_value'], sort=False)
           .size()
           .reset_index(name='cnt')
    )

    mode = (
        counts.sort_values(
            ['profile_id', 'feature_name', 'cnt', 'feature_value'],
            ascending=[True, True, False, True],
            kind='mergesort',
        )
        .drop_duplicates(['profile_id', 'feature_name'])
        .pivot(index='profile_id', columns='feature_name', values='feature_value')
        .reset_index()
    )

    mode.columns = [
        'profile_id' if c == 'profile_id' else f'{prefix}__{c}'
        for c in mode.columns
    ]
    return mode


def coerce_realtime_types(rt_wide: pd.DataFrame) -> pd.DataFrame:
    for c in rt_wide.columns:
        if c == 'profile_id':
            continue

        num = pd.to_numeric(rt_wide[c], errors='coerce')
        non_missing = rt_wide[c].notna().sum()
        if non_missing and num.notna().sum() / non_missing >= 0.95:
            if np.allclose(num.dropna(), np.round(num.dropna())):
                rt_wide[c] = num.astype('Int64')
            else:
                rt_wide[c] = num
        else:
            rt_wide[c] = rt_wide[c].astype('string')
    return rt_wide


np_wide = wide_selected_values(np_long, 'np', WIDE_FEATURES['np'])
fs_wide = wide_selected_values(fs_long, 'fs', WIDE_FEATURES['fs'])
rt_wide = coerce_realtime_types(wide_selected_values(rt_long, 'rt', WIDE_FEATURES['rt']))

print('np_wide:', np_wide.shape)
print('fs_wide:', fs_wide.shape)
print('rt_wide:', rt_wide.shape)

np_wide: (61584, 5)
fs_wide: (35131, 13)
rt_wide: (49380, 3)


In [ ]:
fs_wide.head(13)

## 7. Финальная таблица `profile_flat_for_analysis`

Это основной результат шага.

В ней уже есть:

- минимальный identity-блок в нормализованном виде;
- развернутые `non_processing_features`;
- развернутые `fs_features`;
- развернутые `realtime_features`;
- служебные поля для контроля качества.

`entity_id` оставляем как label/ground truth для анализа, но **не используем как blocking-key**.

In [27]:
flat = (
    profile_base
    .merge(np_wide, on='profile_id', how='left')
    .merge(rt_wide, on='profile_id', how='left')
    .merge(fs_wide, on='profile_id', how='left')
)

for c in [c for c in flat.columns if c.endswith(('__cnt', '__nunique'))]:
    flat[c] = flat[c].fillna(0).astype('int32')

for c in [c for c in flat.columns if c.endswith('__has')]:
    flat[c] = flat[c].astype('boolean').fillna(False).astype(bool)

print(flat.shape)
display(flat.head())

(61927, 32)


,profile_id,event_count,entity_id,entity_size,first_event_at,last_event_at,first_name_norm,last_name_norm,sex_norm,email_norm,email_domain,phone_digits,birthday_ymd,identity_filled_count,np__device,np__geoname_id,np__osfamily,np__subdivision_1_iso_code,rt__geoid,rt__geoname,fs__has_accept_30,fs__has_accept_365,fs__has_account,fs__has_click_30,fs__has_click_365,fs__has_order_30,fs__has_order_365,fs__has_view_90,fs__source_site_30,fs__source_site_365,fs__visited_30,fs__visited_365
0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,1,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,1,2025-11-01 00:27:04.995,2025-11-01 00:27:04.995,анфиса,<NA>,female,lqvxvltxx@mail.ru,mail.ru,<NA>,<NA>,3,smartphone,2013348,android,PRI,2013348,Vladivostok,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1805,1813
1,2403b42b-2420-35c2-ac8e-e6572c2920f6,1,bd0dc28d66040e40ad5303a3ed8f3988973726bb3d0307...,1,2025-11-01 00:38:35.323,2025-11-01 00:38:35.323,надежда,<NA>,female,kstpnq.6751@mail.ru,mail.ru,<NA>,<NA>,3,smartphone,2025339,android,ZAB,2025339,Chita,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6370,NaN,NaN
2,866e538b-a451-32a1-8b2d-06cb3c5ec71b,1,6f2ff507f07d2563b592bd399e193cae866ea4144a261f...,1,2025-11-01 00:47:20.741,2025-11-01 00:47:20.741,<NA>,<NA>,female,vsima16wnf@mail.ru,mail.ru,<NA>,<NA>,2,smartphone,499099,android,SAM,499099,Samara,NaN,1921,1345,NaN,1921,4235,4235,NaN,NaN,4211,NaN,NaN
3,70499ae9-52c6-3286-a763-de039d79533d,1,a6100b59b6ed947d8e7282f79d89297e7c93b5fe4ac057...,1,2025-11-01 00:50:56.909,2025-11-01 00:50:56.909,альберт,<NA>,male,evcqxrz@mail.ru,mail.ru,<NA>,<NA>,3,smartphone,511196,android,PER,511196,Perm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6436,5967,NaN
4,845e13de-2ba8-361b-af0f-5c222bcdf417,1,f894d467f3e527b7f064f15aedcb5d24001fe1df707e0a...,1,2025-11-01 01:11:33.568,2025-11-01 01:11:33.568,кира,<NA>,female,qslmqq.rqppdikroqq.27@mail.ru,mail.ru,<NA>,<NA>,3,smartphone,NaN,ios,NaN,<NA>,<NA>,NaN,1921,1345,NaN,1921,NaN,NaN,NaN,NaN,4211,NaN,NaN


## 8. Feature dictionary

Это словарь развернутых признаков. Он отвечает на вопросы:

- из какого источника пришёл признак;
- сколько токенов было в данных;
- сколько профилей и entity покрывает признак;
- сколько уникальных значений у признака;
- является ли признак только флагом.

In [28]:
def build_feature_dictionary(*tables: pd.DataFrame) -> pd.DataFrame:
    x = pd.concat(
        [t[['source', 'profile_id', 'entity_id', 'feature_name', 'feature_value']] for t in tables if not t.empty],
        ignore_index=True
    )
    n_profiles = x['profile_id'].nunique()

    rows = []
    for (source, feature_name), p in x.groupby(['source', 'feature_name'], sort=False):
        vals = p['feature_value'].dropna().astype(str)
        rows.append({
            'source': source,
            'feature_name': feature_name,
            'token_count': len(p),
            'n_profiles': p['profile_id'].nunique(),
            'n_entities': p['entity_id'].nunique(),
            'profile_share': p['profile_id'].nunique() / n_profiles,
            'n_unique_values': vals.nunique(),
            'is_flag_only': bool(p['feature_value'].isna().all()),
            'top_values': '|'.join(vals.value_counts().head(10).index.tolist()),
        })

    return pd.DataFrame(rows).sort_values(['source', 'token_count'], ascending=[True, False]).reset_index(drop=True)


feature_dictionary = build_feature_dictionary(np_long, fs_long, rt_long)
display(feature_dictionary)

,source,feature_name,token_count,n_profiles,n_entities,profile_share,n_unique_values,is_flag_only,top_values
0,fs_features,has_view_90,217657,3863,3589,0.062635,389,False,3824|3684|3827|3575|6914|1921|3636|6592|6674|3322
1,fs_features,has_account,198144,23777,19923,0.385521,979,False,1345|3589|3210|2275|4880|2492|899|572|277|2054
2,fs_features,has_accept_365,163946,23784,19833,0.385634,664,False,3824|3684|3827|1921|5506|4049|3741|4880|3322|6823
3,fs_features,has_click_365,93619,26161,21909,0.424175,675,False,3824|3684|3827|1921|5506|3741|6823|4049|3322|4880
4,fs_features,source_site_365,84173,31684,26910,0.513725,335,False,4211|6897|6329|6659|4021|6976|3126|2953|3778|4253
5,fs_features,visited_30,50288,23266,20068,0.377236,341,False,1345|1921|572|2275|2986|1805|3657|1721|2925|1696
6,fs_features,has_click_30,32213,14325,12126,0.232266,395,False,3824|3827|3684|1921|6823|5506|6592|3715|6914|3741
7,fs_features,source_site_30,31756,18690,15964,0.303040,227,False,4211|6329|6659|6897|6976|6333|7094|4880|4021|4253
8,fs_features,postman_action_90,30726,17437,14767,0.282724,4,False,smtp_sent|smtp_open|smtp_link|http_sent
9,fs_features,has_accept_30,30540,13773,11524,0.223316,299,False,3824|3827|3684|1921|5506|6823|6592|3715|6914|2531


## 9. Отчёт готовности колонок для blocking / анализа
Думаю его стоит выкинуть!!!

Это не финальный blocking. Это первичная маркировка колонок:

- `candidate_blocking_key` — потенциально годится для blocking;
- `behavior_signal_or_quality_check` — скорее сигнал/контроль качества, а не ключ группировки;
- `sparse_column` — слишком разреженная колонка;
- `high_cardinality_use_with_caution` — высокая кардинальность, использовать осторожно;
- `id_or_label_not_for_blocking` — идентификаторы и label-поля, не использовать как признаки blocking.

In [29]:
def blocking_hint(col: str, missing_share: float, nunique: int, top_share: float) -> str:
    if col in {'profile_id', 'entity_id'}:
        return 'id_or_label_not_for_blocking'
    if col.startswith(('entity_', 'first_event_at', 'last_event_at')):
        return 'metadata_or_label'
    if col.endswith(('__cnt', '__nunique', '__has')):
        return 'behavior_signal_or_quality_check'
    if missing_share <= 0.60 and 2 <= nunique <= 5000 and (pd.isna(top_share) or top_share <= 0.98):
        return 'candidate_blocking_key'
    if missing_share > 0.60:
        return 'sparse_column'
    if nunique <= 1:
        return 'constant_or_near_constant'
    if nunique > 5000:
        return 'high_cardinality_use_with_caution'
    return 'secondary_analysis_column'


def build_column_report(flat: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for c in flat.columns:
        s = flat[c]
        vc = s.value_counts(dropna=True, normalize=True)
        top_share = float(vc.iloc[0]) if len(vc) else np.nan
        missing_share = float(s.isna().mean())
        nunique = int(s.nunique(dropna=True))
        rows.append({
            'column': c,
            'dtype': str(s.dtype),
            'missing_share': missing_share,
            'n_unique': nunique,
            'top_value_share': top_share,
            'blocking_use_hint': blocking_hint(c, missing_share, nunique, top_share),
        })
    return pd.DataFrame(rows).sort_values(['blocking_use_hint', 'missing_share', 'n_unique']).reset_index(drop=True)


column_readiness_report = build_column_report(flat)
display(column_readiness_report.head(40))

,column,dtype,missing_share,n_unique,top_value_share,blocking_use_hint
0,identity_filled_count,int16,0.000000,6,0.636863,candidate_blocking_key
1,event_count,int64,0.000000,18,0.940171,candidate_blocking_key
2,email_domain,string,0.000000,780,0.504675,candidate_blocking_key
3,np__device,object,0.032781,3,0.915906,candidate_blocking_key
4,np__osfamily,object,0.032781,5,0.662921,candidate_blocking_key
5,np__subdivision_1_iso_code,object,0.111438,342,0.034456,candidate_blocking_key
6,np__geoname_id,object,0.191290,1882,0.031130,candidate_blocking_key
7,rt__geoname,string,0.202610,1641,0.031511,candidate_blocking_key
8,rt__geoid,Int64,0.202610,1645,0.031551,candidate_blocking_key
9,fs__source_site_365,object,0.488365,266,0.104311,candidate_blocking_key


## 10. Сохраняем результаты

Основной результат — parquet, потому что CSV теряет типы и плохо подходит для широкой таблицы.

Файлы:

- `profile_flat_for_analysis.parquet` — основная широкая таблица уровня `profile_id`;
- `non_processing_tokens_long.parquet` — длинная таблица токенов `non_processing_features`;
- `fs_tokens_long.parquet` — длинная таблица токенов `fs_features`;
- `realtime_features_long.parquet` — длинная таблица `realtime_features`;
- `feature_dictionary.csv` — словарь признаков;
- `blocking_column_readiness_report.csv` — первичный отчёт пригодности колонок;
- `profile_flat_for_analysis_sample_3000.csv` — лёгкий CSV-сэмпл для просмотра.

In [ ]:
np_long.to_parquet(OUTPUT_DIR / 'non_processing_tokens_long.parquet', index=False)
fs_long.to_parquet(OUTPUT_DIR / 'fs_tokens_long.parquet', index=False)
rt_long.to_parquet(OUTPUT_DIR / 'realtime_features_long.parquet', index=False)

flat.to_parquet(OUTPUT_DIR / 'profile_flat_for_analysis.parquet', index=False)
flat.head(3000).to_csv(OUTPUT_DIR / 'profile_flat_for_analysis_sample_3000.csv', index=False)

feature_dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False)
column_readiness_report.to_csv(OUTPUT_DIR / 'blocking_column_readiness_report.csv', index=False)

summary = {
    'input': str(INPUT_PATH),
    'raw_rows': int(len(raw)),
    'profile_rows': int(len(flat)),
    'entity_count': int(flat['entity_id'].nunique()),
    'flat_columns': int(flat.shape[1]),
    'non_processing_long_rows': int(len(np_long)),
    'fs_long_rows': int(len(fs_long)),
    'realtime_long_rows': int(len(rt_long)),
    'output_dir': str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

## Что дальше

Теперь уже логично делать следующий ноутбук: **candidate generation + pair dataset**.

Но он должен опираться не на сырые массивы, а на подготовленную таблицу `profile_flat_for_analysis.parquet` и отчёт `blocking_column_readiness_report.csv`.

Правильная последовательность:

1. выбрать несколько blocking rules;
2. оценить coverage известных positive pairs по `entity_id`;
3. оценить explosion rate: сколько candidate pairs создаётся;
4. собрать pair dataset;
5. посчитать pair-level признаки сходства.